In [4]:
import pandas as pd

deployments = pd.read_csv("../data/raw/deployments.csv")
meta_votes = deployments.groupby("platform")["votes_count"].sum()
print(meta_votes)
print("Total declared:", meta_votes.sum())

platform
aragon         26833
daohaus        51129
daostack       12331
realms       1067333
snapshot    64697595
tally        1440390
Name: votes_count, dtype: int64
Total declared: 67295611


In [ ]:
chunk_counts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["proposal_id", "platform"],
                         dtype={"proposal_id": str},
                         chunksize=1_000_000):
    snapshot_votes = chunk[chunk["platform"] == "snapshot"]
    chunk_counts.append(snapshot_votes["proposal_id"].value_counts())

actual_votes = pd.concat(chunk_counts).groupby(level=0).sum()
print("Snapshot proposals with votes:", len(actual_votes))
print("Max votes in a single proposal:", actual_votes.max())
print(actual_votes.value_counts().head(10))

C:\Users\alexs\AppData\Local\Temp\ipykernel_10076\1562435163.py:2: DtypeWarning: Columns (1: proposal_id) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv("../data/raw/votes.csv",


Snapshot proposals with votes: 198297
Max votes in a single proposal: 6000
count
1     83090
2     21384
3     10442
4      6568
5      5392
6      4010
7      3505
8      2960
10     2732
9      2656
Name: count, dtype: int64


In [5]:
proposals = pd.read_csv("../data/raw/proposals.csv",
                        dtype={"proposal_id": str, "platform_deployment_id": str})
snapshot_declared = (proposals[proposals["platform"] == "snapshot"]
                     .set_index("proposal_id")["votes_count"])

comparison = pd.DataFrame({"declared": snapshot_declared,
                           "actual": actual_votes}).fillna(0)
comparison["missing"] = comparison["declared"] - comparison["actual"]

capped = comparison["actual"] == 6000
empty = comparison["actual"] == 0

print("Proposals:", len(comparison))
print("Declared votes:", comparison["declared"].sum())
print("Actual votes:", comparison["actual"].sum())
print("Proposals capped at 6000:", capped.sum())
print("Missing votes in capped proposals:", comparison.loc[capped, "missing"].sum())
print("Proposals with 0 actual votes but declared > 0:",
      (empty & (comparison["declared"] > 0)).sum())
print("Missing votes in those proposals:", comparison.loc[empty, "missing"].sum())

Proposals: 244898
Declared votes: 65716576.0
Actual votes: 8883118.0
Proposals capped at 6000: 13
Missing votes in capped proposals: 38472.0
Proposals with 0 actual votes but declared > 0: 17776
Missing votes in those proposals: 56800302.0


In [6]:
snapshot_props = proposals[proposals["platform"] == "snapshot"].copy()
snapshot_props["dao"] = snapshot_props["deployment_id"].fillna(
    snapshot_props["platform_deployment_id"])
snapshot_props["year"] = pd.to_datetime(snapshot_props["date"], errors="coerce").dt.year
snapshot_props["in_votes_csv"] = snapshot_props["proposal_id"].isin(actual_votes.index)

by_year = (snapshot_props.groupby(["year", "in_votes_csv"])["votes_count"]
           .sum().unstack(fill_value=0))
print(by_year)

missing_rows = snapshot_props[~snapshot_props["in_votes_csv"]
                              & (snapshot_props["votes_count"] > 0)]
print("DAOs affected:", missing_rows["dao"].nunique())
print(missing_rows.groupby("dao")["votes_count"].sum()
      .sort_values(ascending=False).head(10))

in_votes_csv       False      True 
year                               
2020             55361.0    55357.0
2021           1140859.0  1490862.0
2022           2937908.0  2468847.0
2023          45802500.0  2649067.0
2024           6640127.0  1509725.0
2025            223547.0   742416.0
DAOs affected: 2204
dao
magicappstore.eth         22223273.0
stgdao.eth                18198616.0
arbitrumfoundation.eth     5274117.0
linea-build.eth            3135558.0
aavedao.eth                2558198.0
opcollective.eth           1121529.0
whitelist1.eth              326823.0
cakevote.eth                305051.0
gmx.eth                     277768.0
syncswapxyz.eth             220917.0
Name: votes_count, dtype: float64


In [7]:
snapshot_props["actual"] = snapshot_props["proposal_id"].map(actual_votes).fillna(0)

dao_coverage = snapshot_props.groupby("dao").agg(
    proposals=("proposal_id", "size"),
    proposals_with_votes=("in_votes_csv", "sum"),
    declared=("votes_count", "sum"),
    actual=("actual", "sum"),
)
dao_coverage["coverage"] = dao_coverage["actual"] / dao_coverage["declared"]

top_missing = missing_rows.groupby("dao")["votes_count"].sum().nlargest(10).index
print(dao_coverage.loc[top_missing])

                        proposals  proposals_with_votes    declared    actual  \
dao                                                                             
magicappstore.eth             323                    48  22310132.0   86859.0   
stgdao.eth                    155                    52  18359814.0  146887.0   
arbitrumfoundation.eth        394                   108   5652089.0  377972.0   
linea-build.eth                 2                     0   3135558.0       0.0   
aavedao.eth                   954                   696   3223427.0  665229.0   
opcollective.eth               93                     5   1144456.0   22927.0   
whitelist1.eth                106                    81    454128.0  127305.0   
cakevote.eth                 4662                  3671    959487.0  654449.0   
gmx.eth                        66                    52    314157.0   36389.0   
syncswapxyz.eth                 1                     0    220917.0       0.0   

                        cov

In [9]:
present = comparison[comparison["actual"] > 0]
missing = comparison[(comparison["actual"] == 0) & (comparison["declared"] > 0)]

print("Max declared votes among present proposals:", present["declared"].max())
print("Present proposals with declared > 6000:", (present["declared"] > 6000).sum())
print("Missing proposals with declared > 6000:", (missing["declared"] > 6000).sum())
print(missing["declared"].describe())

large_missing = missing["declared"] > 6000
print("Missing votes, proposals > 6000:", missing.loc[large_missing, "declared"].sum())
print("Missing votes, proposals <= 6000:", missing.loc[~large_missing, "declared"].sum())

Max declared votes among present proposals: 20935.0
Present proposals with declared > 6000: 13
Missing proposals with declared > 6000: 1040
count    1.777600e+04
mean     3.195337e+03
std      3.131525e+04
min      1.000000e+00
25%      4.000000e+00
50%      1.600000e+01
75%      6.100000e+01
max      2.764428e+06
Name: declared, dtype: float64
Missing votes, proposals > 6000: 54749561.0
Missing votes, proposals <= 6000: 2050741.0


In [10]:
tally_parts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["vote_id", "platform"],
                         dtype={"vote_id": str},
                         chunksize=1_000_000):
    tally_parts.append(chunk.loc[chunk["platform"] == "tally", "vote_id"])

tally_ids = pd.concat(tally_parts)
print("Tally vote rows:", len(tally_ids))
print("Distinct vote_id:", tally_ids.nunique())

Tally vote rows: 1597604
Distinct vote_id: 1597604


In [11]:
tally_parts = []
for chunk in pd.read_csv("../data/raw/votes.csv",
                         usecols=["deployment_id", "proposal_id", "voter", "platform"],
                         dtype=str, chunksize=1_000_000):
    tally_parts.append(chunk[chunk["platform"] == "tally"])

tally_votes = pd.concat(tally_parts)
tally_votes["deployment_id"] = tally_votes["deployment_id"].str.lower()

print("Repeated (proposal_id, voter) pairs:",
      tally_votes.duplicated(subset=["proposal_id", "voter"]).sum())

tally_declared = (deployments[deployments["platform"] == "tally"]
                  .set_index("deployment_id")["votes_count"])
tally_actual = tally_votes.groupby("deployment_id").size()

tally_cmp = pd.DataFrame({"declared": tally_declared,
                          "actual": tally_actual}).fillna(0)
tally_cmp["extra"] = tally_cmp["actual"] - tally_cmp["declared"]
not_in_catalogue = ~tally_cmp.index.isin(tally_declared.index)

print("Votes in deployments missing from deployments.csv:",
      tally_cmp.loc[not_in_catalogue, "actual"].sum())
print("Deployments with more votes than declared:", (tally_cmp["extra"] > 0).sum())
print("Deployments with fewer votes than declared:", (tally_cmp["extra"] < 0).sum())
print(tally_cmp.sort_values("extra", ascending=False).head(5))

Repeated (proposal_id, voter) pairs: 559
Votes in deployments missing from deployments.csv: 0.0
Deployments with more votes than declared: 101
Deployments with fewer votes than declared: 934
                                                    declared    actual  \
deployment_id                                                            
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...    240303  351098.0   
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...    257389  283434.0   
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...      6258   23173.0   
eip155:7887:0x27926a991bb0193bf5b679bdb6cb3d3b6...     37426   50469.0   
eip155:324:0x76705327e682f2d96943280d99464ab612...     50216   54800.0   

                                                       extra  
deployment_id                                                 
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...  110795.0  
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...   26045.0  
eip155:324:0xb83ff6501214ddf40c91c9565

In [12]:
above = tally_cmp.loc[tally_cmp["extra"] > 0, "extra"].sum()
below = tally_cmp.loc[tally_cmp["extra"] < 0, "extra"].sum()
print("Votes above declared:", above)
print("Votes below declared:", below)

top_extra = tally_cmp.sort_values("extra", ascending=False).head(5).index
print(deployments.set_index("deployment_id").loc[top_extra, ["name", "unique_voters"]])

Votes above declared: 190512.0
Votes below declared: -33298.0
                                                                          name  \
deployment_id                                                                    
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...                        test   
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...               arbitrum core   
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...             zktokengovernor   
eip155:7887:0x27926a991bb0193bf5b679bdb6cb3d3b6...  kinto (proto governance) 2   
eip155:324:0x76705327e682f2d96943280d99464ab612...          zkprotocolgovernor   

                                                    unique_voters  
deployment_id                                                      
eip155:10:0xcdf27f107725988f2261ce2256bdfcde8b3...        78724.0  
eip155:42161:0xf07ded9dc292157749b6fd268e37df6e...       112713.0  
eip155:324:0xb83ff6501214ddf40c91c9565d095400f3...         3012.0  
eip155:7887:0x27926a991